# Tema 08 - Laboratorio 03: Operaciones con CSV

**Objetivo.** Escribir y leer CSV con writer, DictWriter, reader y DictReader, y normalizar datos desde servicios3.csv.

> **Uso recomendado:** ejecuta las celdas en orden. Cada bloque representa un paso del laboratorio y mantiene el estado necesario para los pasos posteriores.

## Preparación de datos

> Comentario: esta celda garantiza que `servicios3.csv` existe para poder ejecutar el laboratorio aunque el notebook se use fuera del repositorio.

In [ ]:
from pathlib import Path
carpeta = Path("Tema08/data")
carpeta.mkdir(parents=True, exist_ok=True)
ruta_csv3 = carpeta / "servicios3.csv"
if not ruta_csv3.exists():
    ruta_csv3.write_text('servicio,puerto,estado\n SSH , 22 , OK \n NGINX , 443 , ok \n api , 8080 , ERROR \n BACKUP ,no_numero, ERROR \n dns , 53 , OK \n  postgresql  ,5432, ok\n', encoding="utf-8")
print("Fichero preparado:", ruta_csv3)
print(ruta_csv3.read_text(encoding="utf-8"))

## Paso 1. 1. Escribir CSV con csv.writer

> Comentario: `csv.writer` permite escribir filas simples como listas de valores.

In [ ]:
"""
Tema 8 - Operaciones de entrada-salida
Laboratorio 3: operaciones con CSV.

Objetivo:
    Escribir y leer CSV con writer, DictWriter, reader y DictReader.
    Normalizar datos desde data/servicios3.csv.
"""
import csv
from pathlib import Path

carpeta = Path("Tema08/data")
carpeta.mkdir(parents=True, exist_ok=True)

print("=== 1. Escribir CSV con csv.writer ===")
ruta_csv1 = carpeta / "servicios1.csv"
with open(ruta_csv1, "w", newline="", encoding="utf-8") as filecsv:
    escritor = csv.writer(filecsv, delimiter=",")
    escritor.writerow(["servicio", "puerto", "estado"])
    escritor.writerow(["ssh", 22, "OK"])
    escritor.writerow(["http", 80, "OK"])
    escritor.writerow(["https", 443, "OK"])
print("CSV creado:", ruta_csv1)

## Paso 2. 2. Escribir CSV con csv.DictWriter

> Comentario: `csv.DictWriter` escribe diccionarios respetando los nombres definidos en `fieldnames`.

In [ ]:
print("\n=== 2. Escribir CSV con csv.DictWriter ===")
ruta_csv2 = carpeta / "servicios2.csv"
campos = ["servicio", "puerto", "estado"]
filas = [
    {"servicio": "ssh", "puerto": 22, "estado": "OK"},
    {"servicio": "http", "puerto": 80, "estado": "OK"},
    {"servicio": "postgresql", "puerto": 5432, "estado": "OK"},
]
with open(ruta_csv2, "w", newline="", encoding="utf-8") as filecsv:
    escritor = csv.DictWriter(filecsv, fieldnames=campos, delimiter=",")
    escritor.writeheader()
    escritor.writerows(filas)
print("CSV creado:", ruta_csv2)

## Paso 3. 3. Leer CSV con csv.reader

> Comentario: `csv.reader` devuelve cada fila como lista. La cabecera se consume con `next()`.

In [ ]:
print("\n=== 3. Leer CSV con csv.reader ===")
with open(ruta_csv1, "r", newline="", encoding="utf-8") as filecsv:
    lector = csv.reader(filecsv, delimiter=",")
    cabecera = next(lector)
    print("Cabecera:", cabecera)
    for fila in lector:
        servicio, puerto, estado = fila
        print(f"Servicio: {servicio} -> {puerto}: {estado}")

## Paso 4. 4. Leer CSV con csv.DictReader

> Comentario: `csv.DictReader` devuelve cada fila como diccionario. Los campos numéricos siguen llegando como texto y deben convertirse.

In [ ]:
print("\n=== 4. Leer CSV con csv.DictReader ===")
with open(ruta_csv2, "r", newline="", encoding="utf-8") as filecsv:
    lector = csv.DictReader(filecsv, delimiter=",")
    for fila in lector:
        puerto = int(fila["puerto"])
        print(f"Servicio: {fila['servicio']} -> Puerto {puerto} [{fila['estado']}]")

## Paso 5. 5. Normalizar datos desde servicios3.csv

> Comentario: Se normalizan espacios, mayúsculas y minúsculas. Los puertos no convertibles se separan como registros rechazados.

In [ ]:
print("\n=== 5. Normalizar datos desde servicios3.csv ===")
ruta_csv3 = carpeta / "servicios3.csv"
servicios_validos = []
servicios_rechazados = []

try:
    with open(ruta_csv3, "r", newline="", encoding="utf-8") as filecsv:
        lector = csv.DictReader(filecsv, delimiter=",")
        for fila in lector:
            servicio = fila["servicio"].strip().lower()
            puerto_txt = fila["puerto"].strip()
            estado = fila["estado"].strip().upper()
            try:
                puerto = int(puerto_txt)
            except ValueError:
                servicios_rechazados.append({"servicio": servicio, "puerto": puerto_txt, "motivo": "puerto no numérico"})
            else:
                servicios_validos.append({"servicio": servicio, "puerto": puerto, "estado": estado})
except FileNotFoundError:
    print(f"[CRÍTICO] No existe el fichero esperado: {ruta_csv3}")

print("Servicios válidos normalizados:")
for servicio in servicios_validos:
    print(servicio)
print("Servicios rechazados:")
for servicio in servicios_rechazados:
    print(servicio)

## Paso 6. 6. Guardar CSV normalizado

> Comentario: Los registros válidos se guardan en un nuevo CSV normalizado.

In [ ]:
print("\n=== 6. Guardar CSV normalizado ===")
ruta_normalizada = carpeta / "servicios3_normalizado.csv"
with open(ruta_normalizada, "w", newline="", encoding="utf-8") as filecsv:
    campos = ["servicio", "puerto", "estado"]
    escritor = csv.DictWriter(filecsv, fieldnames=campos)
    escritor.writeheader()
    escritor.writerows(servicios_validos)
print("CSV normalizado guardado en:", ruta_normalizada)